# Distribuzione della proprieta target

Questo notebook serve a guardare come e' distribuita la proprieta' scelta nel dataset. I dati vengono letti da MongoDB, il campo `properties` viene portato in colonne normali e vengono tenuti solo i campioni utilizzabili per il target selezionato.

Il grafico finale mostra la distribuzione dei valori e aiuta a capire se il target e' concentrato in un intervallo ristretto, se ci sono gruppi distinti o se sono presenti valori estremi. Per analizzare un'altra proprieta basta cambiare la variabile `target` nella cella di configurazione.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Impostazioni estetiche dei grafici
sns.set(style="whitegrid", context="talk")

PROJECT_NAME = "polymer_prediction"
ROOT = Path.cwd()
while ROOT != ROOT.parent and ROOT.name != PROJECT_NAME:
    ROOT = ROOT.parent
if ROOT.name != PROJECT_NAME:
    raise RuntimeError(f"Root del progetto {PROJECT_NAME!r} non trovata")

In [ ]:
from dotenv import load_dotenv
from pymongo import MongoClient
import os

load_dotenv(ROOT / ".env")
MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db = client["PolymerPrediction"]
collection = db["biceranoPolymers"]
target = "rubber_cte"

In [ ]:
docs = list(collection.find({}))
df = pd.DataFrame(docs)

# Porto le proprieta annidate in colonne normali
prop_df = pd.json_normalize(df["properties"])

# Integro i dati nel dataframe
df = pd.concat([df.drop(columns=["properties"]), prop_df], axis=1)

# rimuovo righe senza SMILES, Tg o densità o con SMILES malformato
df = df.dropna(subset=["smiles", target])
df = df[~df["smiles"].str.contains(r"\[\[", na=False)].copy()
df.describe()

In [ ]:
plt.figure(figsize=(10, 6))

# Istogramma con KDE (density plot)
sns.histplot(
    df[target],
    kde=True,
    bins=20,      # puoi adattare i bins
    color="cornflowerblue"
)

plt.xlabel(target, fontsize=14)
plt.ylabel("Count", fontsize=14)
plt.title(f"Distribuzione di {target} nel dataset Bicerano (HF)", fontsize=16)
plt.tight_layout()
plt.show()